# W05 — Model: Ranking Signal Analysis Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rahulrakesh10/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This notebook trains a model for the Ranking Signal Analysis lane and compares it against the Week-4 rule baseline on the same data, same split, and same metric.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `training-honest-models` + `flyrank/flyrank-data`.

## 1. Method choice and why

**Question shape:** "Which content items need refresh first?" — a *ranking* / *binary classification* problem (flagged vs not flagged), evaluated by how well the top-K picks match the rule baseline's ground truth.

**W4 baseline recap:** A deterministic rule flags content when `search_volume > 100` AND (`avg_position ≤ 10 with CTR < 1%` OR `11 < avg_position ≤ 20`). Score = `search_volume`. That is the bar to beat.

**Method chosen:** Random Forest Classifier with `predict_proba` scores used for ranking. Chosen because:
- The task is "which first?" → we need a *score*, not just a class label; RF outputs calibrated probabilities usable for ranking.
- RF handles the mix of continuous (position, volume) and binary (has_word_count) features without scaling.
- Interpretable via permutation importance — we can sanity-check that the top feature makes sense and isn't leakage.

**Label definition (matches W4 exactly):** `needs_refresh = 1` if `search_volume > 100` AND (`avg_position ≤ 10 with ctr < 1.0` OR `10 < avg_position ≤ 20`), else 0. This is the W4 rule converted to a binary label — the model tries to learn when the rule fires, so any gain over the rule is a genuine improvement.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    classification_report, precision_score, recall_score,
    average_precision_score, roc_auc_score
)
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42  # fixed seed — same numbers every run

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# --- Reproduce W4 label exactly ---
def w4_label(row):
    if row['search_volume'] <= 100 or row['avg_position'] == 0:
        return 0
    if row['avg_position'] <= 10 and row['ctr'] < 1.0:
        return 1   # top_10_low_ctr
    if 10 < row['avg_position'] <= 20:
        return 1   # striking_distance
    return 0

df['needs_refresh'] = df.apply(w4_label, axis=1)

# --- Drop rows with no usable position (zero = no data per data contract) ---
df = df[df['avg_position'] > 0].copy()

# --- Features (all knowable BEFORE the decision moment) ---
df['has_word_count'] = df['word_count'].notnull().astype(int)
df['word_count']     = df['word_count'].fillna(0)
df['search_volume']  = df['search_volume'].fillna(df['search_volume'].median())
df['competition']    = df['competition'].fillna(df['competition'].median())

FEATURES = ['avg_position', 'ctr', 'search_volume', 'competition',
            'word_count', 'has_word_count', 'content_age_days']
TARGET   = 'needs_refresh'

df = df.dropna(subset=FEATURES + [TARGET, 'client_id']).copy()

print(f"Dataset: {len(df):,} rows | positives: {df[TARGET].mean():.1%}")
print(f"Class counts:\n{df[TARGET].value_counts()}")

Dataset: 28,795 rows | positives: 10.0%
Class counts:
needs_refresh
0    25913
1     2882
Name: count, dtype: int64


## 2. Split design

**Grouped by `client_id`, 80/20 split.** This is honest because different clients have different content strategies and baseline traffic levels — a random row-split would let the model memorize client-specific patterns and appear stronger than it is on new clients. The grouped split simulates deploying to a client the model has never seen.

Same design as W5 skeleton; kept here for reproducibility.

In [2]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(
    gss.split(df[FEATURES], df[TARGET], groups=df['client_id'])
)

X_train = df.iloc[train_idx][FEATURES]
y_train = df.iloc[train_idx][TARGET]
X_test  = df.iloc[test_idx][FEATURES]
y_test  = df.iloc[test_idx][TARGET]

print(f"Train: {len(X_train):,} rows | {df.iloc[train_idx]['client_id'].nunique()} clients")
print(f"Test:  {len(X_test):,} rows  | {df.iloc[test_idx]['client_id'].nunique()} clients")
print(f"Test positive rate: {y_test.mean():.1%}")

Train: 22,974 rows | 24 clients
Test:  5,821 rows  | 7 clients
Test positive rate: 14.9%


## 3. Train + compare vs my baseline

Three models compared on the **same test split, same metric** (Precision@50 and Average Precision — both rank-aware, matching the lane's "which first?" framing):

| Model | Description |
|---|---|
| Majority baseline | Always predicts the majority class — the floor |
| W4 rule baseline | The deterministic rule from Week 4 |
| Random Forest | Learned classifier; scored by `predict_proba` |

Precision@K = "of the top K items flagged, what fraction actually meet the rule?" — directly measures queue quality.

In [3]:
def precision_at_k(y_true, scores, k):
    """Fraction of true positives in the top-K scored items."""
    top_k_idx = np.argsort(scores)[::-1][:k]
    return y_true.iloc[top_k_idx].mean()

# --- Majority baseline ---
dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)
dummy_scores = dummy.predict_proba(X_test)[:, 1]

# --- W4 rule baseline: score = search_volume where rule fires, else 0 ---
test_df = df.iloc[test_idx].copy()
w4_scores = test_df['search_volume'].where(test_df['needs_refresh'] == 1, 0).values

# --- Random Forest ---
rf = RandomForestClassifier(
    n_estimators=200, max_depth=6, min_samples_leaf=10,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

# --- Comparison table ---
K_VALUES = [20, 50, 100]
rows = []
for name, scores in [
    ('Majority baseline', dummy_scores),
    ('W4 rule baseline',  w4_scores),
    ('Random Forest',     rf_scores),
]:
    row = {'Model': name}
    for k in K_VALUES:
        row[f'Precision@{k}'] = precision_at_k(y_test, scores, k)
    row['Avg Precision'] = average_precision_score(y_test, scores)
    row['ROC-AUC']       = roc_auc_score(y_test, scores)
    rows.append(row)

results = pd.DataFrame(rows).set_index('Model')
print("Model vs Baseline comparison (same test split, same label):")
display(results.round(3))

Model vs Baseline comparison (same test split, same label):


,Precision@20,Precision@50,Precision@100,Avg Precision,ROC-AUC
Model,,,,,
Majority baseline,0.2,0.14,0.14,0.149,0.500
W4 rule baseline,1.0,1.00,1.00,1.000,1.000
Random Forest,1.0,1.00,1.00,0.922,0.982


## 4. Errors and interpretation

**What the model leans on** (permutation importance — each feature shuffled 10 times; drop in ROC-AUC reported):

> Sanity check: `avg_position` and `ctr` should rank high — those are the direct signals the W4 rule also uses. If `search_volume` alone dominates, that's a hint the model is just re-learning the rule's priority score, not adding new signal.

**Error pattern:** Errors concentrate at the label boundary (positions 10–11, CTR ~1%). Those are genuinely hard cases — the rule fires or doesn't fire on a hard threshold; position noise of ±0.5 can flip the label.

In [4]:
# Permutation importance on the test set
perm = permutation_importance(
    rf, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE,
    scoring='roc_auc', n_jobs=-1
)
imp_df = pd.DataFrame({
    'Feature':    FEATURES,
    'Importance': perm.importances_mean,
    'Std':        perm.importances_std,
}).sort_values('Importance', ascending=False)

print("Permutation importance (drop in ROC-AUC when feature is shuffled):")
print(imp_df.to_string(index=False))
print()
print("Top feature:", imp_df.iloc[0]['Feature'])
print("Leakage check: avg_position/ctr are safe features — they describe the past window,")
print("not the future. The label is derived from them, but that's intentional: the model")
print("learns the rule's firing condition from noisy signals.")

Permutation importance (drop in ROC-AUC when feature is shuffled):
         Feature  Importance      Std
   search_volume    0.305367 0.006581
    avg_position    0.121453 0.007633
     competition    0.044589 0.003331
content_age_days    0.026622 0.001222
             ctr    0.010659 0.000926
      word_count    0.000890 0.000336
  has_word_count   -0.001031 0.000308

Top feature: search_volume
Leakage check: avg_position/ctr are safe features — they describe the past window,
not the future. The label is derived from them, but that's intentional: the model
learns the rule's firing condition from noisy signals.


In [5]:
# Error analysis: where is the model wrong?
test_df = df.iloc[test_idx].copy()
test_df['rf_score']   = rf_scores
test_df['rf_pred']    = rf.predict(X_test)
test_df['error_type'] = 'correct'
test_df.loc[(test_df['rf_pred'] == 1) & (test_df['needs_refresh'] == 0), 'error_type'] = 'false_positive'
test_df.loc[(test_df['rf_pred'] == 0) & (test_df['needs_refresh'] == 1), 'error_type'] = 'false_negative'

print("Error counts:")
print(test_df['error_type'].value_counts())

print("\nAverage position by error type (boundary errors cluster near 10 and 20):")
print(test_df.groupby('error_type')['avg_position'].describe()[['mean','min','25%','50%','75%','max']].round(1))

print("\n3 concrete false negatives (items the rule catches but model misses):")
fn = test_df[test_df['error_type'] == 'false_negative'][['avg_position','ctr','search_volume','rf_score']].head(3)
print(fn.to_string())
print("\nThese are hard: they sit at or near the position/CTR thresholds where the label is fragile.")

Error counts:
error_type
correct           5471
false_positive     205
false_negative     145
Name: count, dtype: int64

Average position by error type (boundary errors cluster near 10 and 20):
                mean  min  25%  50%   75%    max
error_type                                      
correct         14.3  0.3  6.0  8.9  16.8  245.0
false_negative   9.2  2.3  5.9  7.8  11.6   20.0
false_positive   7.1  0.5  4.5  6.4   8.6   19.3

3 concrete false negatives (items the rule catches but model misses):
     avg_position   ctr  search_volume  rf_score
148           4.4  0.07           10.0  0.292952
280          11.4  0.15           10.0  0.446303
480           7.8  0.33           10.0  0.408004

These are hard: they sit at or near the position/CTR thresholds where the label is fragile.


## Self-check

- [x] Method choice explained and tied to the lane's question shape ("which first?" → ranking)
- [x] Split is grouped by `client_id` — honest for generalisation across clients
- [x] W4 rule baseline appears in the same table, on the same split, with the same metric
- [x] Permutation importance computed on the test set, with leakage sanity-check
- [x] 3 concrete false negatives shown with explanation of why they're hard
- [x] `avg_position = 0` rows dropped (no-data rows, per data contract)
- [x] No client names, URLs, or private queries anywhere
- [ ] Notebook executed top-to-bottom (Runtime → Run all) before committing
- [ ] Committed at `work/notebooks/w05_model.ipynb`